In [1]:
!pip install torch torchvision torchaudio
!pip install pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv
!pip install torch-geometric
!pip install numpy pandas scikit-learn hyperopt openpyxl

ERROR: Could not find a version that satisfies the requirement pyg-lib (from versions: none)
ERROR: No matching distribution found for pyg-lib
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.9 MB/s eta 0:00:00


In [2]:
import torch
import torch_geometric
import sklearn
import hyperopt
import pandas

print("Todo OK")

Todo OK


In [3]:
import numpy as np
import pandas as pd

def generate_spatial_soil_dataset(n_samples=1000, n_features=47, output_file="large_synthetic_soil_dataset.csv", seed=42):
    """
    Genera un dataset sintético para estimación de SOC (Soil Organic Carbon)
    incorporando fuerte autocorrelación espacial para probar modelos GNN.
    """
    np.random.seed(seed)
    print(f"Generando {n_samples} muestras espaciales...")

    # 1. Bounding Box (Aprox. Ecuador continental)
    lat_min, lat_max = -5.0, 1.5
    lon_min, lon_max = -81.0, -75.0

    lats = np.random.uniform(lat_min, lat_max, n_samples)
    lons = np.random.uniform(lon_min, lon_max, n_samples)

    # 2. Generar Features (47 variables simulando espectroscopía/clima)
    # Generamos matrices de covarianza aleatorias para que los features estén correlacionados
    features = np.zeros((n_samples, n_features))
    for i in range(n_features):
        # Algunos features dependen levemente de la lat/lon para simular gradientes climáticos
        lat_effect = np.sin(lats * 2.0) * np.random.uniform(0, 0.5)
        lon_effect = np.cos(lons * 2.0) * np.random.uniform(0, 0.5)
        base_noise = np.random.normal(0, 1, n_samples)
        features[:, i] = base_noise + lat_effect + lon_effect

    # 3. Construir la variable dependiente (SOC)
    # 3a. Efecto Espacial Fuerte (Autocorrelación bidimensional)
    # Simulamos "zonas" ecológicas usando funciones trigonométricas combinadas
    spatial_effect = (
        np.sin(lats * 3.0) * np.cos(lons * 3.0) * 2.0 +
        np.sin((lats + lons) * 5.0) * 1.5
    )

    # 3b. Efecto de los Features (Ground Truth)
    # Elegimos 5 features al azar para que tengan una relación fuerte con el SOC
    important_feats_idx = np.random.choice(n_features, size=5, replace=False)
    weights = np.random.uniform(-1.5, 2.5, size=5)

    feature_effect = np.zeros(n_samples)
    for idx, w in zip(important_feats_idx, weights):
        feature_effect += features[:, idx] * w

    # Añadimos no-linealidad a un feature
    feature_effect += (features[:, important_feats_idx[0]] ** 2) * 0.5

    # 3c. Componente Base y Ruido Irreducible
    base_soc = 2.5 # Valor medio base
    noise = np.random.normal(0, 0.4, n_samples)

    # Suma total para obtener el SOC
    soc_raw = base_soc + spatial_effect + feature_effect + noise

    # Asegurar que el SOC sea positivo (recortamos colas negativas raras)
    soc_final = np.clip(soc_raw, a_min=0.1, a_max=None)

    # 4. Ensamblar y Guardar DataFrame
    columns = ["lat", "lon"] + [f"feat_{i+1}" for i in range(n_features)] + ["SOC"]
    data_matrix = np.column_stack((lats, lons, features, soc_final))

    df = pd.DataFrame(data_matrix, columns=columns)

    # Redondeamos para simular precisión de instrumentos
    df["lat"] = df["lat"].round(4)
    df["lon"] = df["lon"].round(4)
    for i in range(1, n_features + 1):
        df[f"feat_{i}"] = df[f"feat_{i}"].round(4)
    df["SOC"] = df["SOC"].round(4)

    df.to_csv(output_file, index=False)
    print(f"✅ Dataset guardado con éxito en: {output_file}")
    print(f"Resumen SOC: Media={df['SOC'].mean():.2f}, Min={df['SOC'].min():.2f}, Max={df['SOC'].max():.2f}")

# Ejecutar generador
generate_spatial_soil_dataset(n_samples=1000, output_file="large_synthetic_soil_dataset.csv")

Generando 1000 muestras espaciales...
✅ Dataset guardado con éxito en: large_synthetic_soil_dataset.csv
Resumen SOC: Media=3.44, Min=0.10, Max=16.55


# Entonque inductivo GraphSage

In [5]:
import os
import json
import random
import multiprocessing
from dataclasses import dataclass
from typing import Dict, Any, List, Optional

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import dropout_edge

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval

# Uso al 100% de la CPU para operaciones de tensores
torch.set_num_threads(multiprocessing.cpu_count())

print("Cargando pipeline GraphSAGE INDUCTIVO (Grafos Aislados + Log-Target + Regularizado)...")

# =========================
# Reproducibility & Metrics
# =========================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def rmse(y_true, y_pred) -> float: return float(np.sqrt(mean_squared_error(y_true, y_pred)))
def safe_mape(y_true, y_pred) -> float:
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mask = y_true != 0
    if not np.any(mask): return float("inf")
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0)

def make_preprocessor(normalizer: str):
    imputer = SimpleImputer(strategy="median")
    if normalizer == "standard": return imputer, StandardScaler()
    elif normalizer == "minmax": return imputer, MinMaxScaler()
    elif normalizer == "robust": return imputer, RobustScaler()
    raise ValueError(normalizer)

# =========================
# Topological Mapping
# =========================
def knn_edge_index_haversine(lat_deg: np.ndarray, lon_deg: np.ndarray, k: int) -> torch.Tensor:
    from sklearn.neighbors import NearestNeighbors
    n = len(lat_deg)
    if n <= 1: return torch.empty((2, 0), dtype=torch.long)
    k_eff = min(k, n - 1)
    coords_rad = np.deg2rad(np.column_stack([lat_deg, lon_deg]))
    # n_jobs=-1 usa todos los nucleos de la CPU
    nn_model = NearestNeighbors(n_neighbors=k_eff + 1, metric="haversine", algorithm="ball_tree", n_jobs=-1)
    nn_model.fit(coords_rad)
    _, indices = nn_model.kneighbors(coords_rad)
    src, dst = [], []
    for i in range(n):
        for j in indices[i, 1:]:
            src.append(int(j)), dst.append(int(i))
    return torch.tensor([src, dst], dtype=torch.long)

def build_pyg_data(X: np.ndarray, y: np.ndarray, lat: np.ndarray, lon: np.ndarray, k_graph: int) -> Data:
    edge_index = knn_edge_index_haversine(lat, lon, k_graph)
    return Data(x=torch.from_numpy(X.astype(np.float32)), y=torch.from_numpy(y.astype(np.float32)), edge_index=edge_index)

# =========================
# Model: GraphSAGE Regressor (Inductivo)
# =========================
class GraphSAGERegressorOptimal(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, num_layers: int, dropout: float, residual: bool):
        super().__init__()
        num_layers = max(2, num_layers)
        self.dropout = dropout
        self.residual = residual
        self.edge_dropout_prob = 0.15 # DropEdge

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.res_proj = nn.ModuleList() if residual else None

        self.convs.append(SAGEConv(in_dim, hidden_dim))
        self.norms.append(nn.LayerNorm(hidden_dim))
        if residual: self.res_proj.append(nn.Linear(in_dim, hidden_dim))

        for _ in range(1, num_layers):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.norms.append(nn.LayerNorm(hidden_dim))
            if residual: self.res_proj.append(nn.Linear(hidden_dim, hidden_dim))

        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        if self.training:
            edge_index, _ = dropout_edge(edge_index, p=self.edge_dropout_prob, force_undirected=True)

        h = x
        for li, conv in enumerate(self.convs):
            h_in = h
            h = F.dropout(h, p=self.dropout, training=self.training)
            h = conv(h, edge_index)
            h = self.norms[li](h)
            if self.residual: h = h + self.res_proj[li](h_in)
            h = F.gelu(h)

        return self.mlp(h).squeeze(-1)

# =========================
# Training with Early Stopping
# =========================
@dataclass
class TrainCfg:
    epochs: int = 300
    patience: int = 35
    lr: float = 1e-3
    weight_decay: float = 1e-4
    loss: str = "huber"
    huber_delta: float = 1.0
    grad_clip: float = 5.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

def loss_fn(pred: torch.Tensor, y: torch.Tensor, cfg: TrainCfg) -> torch.Tensor:
    if cfg.loss == "mse": return F.mse_loss(pred, y)
    if cfg.loss == "huber": return F.smooth_l1_loss(pred, y, beta=cfg.huber_delta)

def train_fullbatch_with_es(data: Data, model: nn.Module, cfg: TrainCfg) -> nn.Module:
    model = model.to(cfg.device)
    data = data.to(cfg.device)

    n = data.x.size(0)
    rng = np.random.default_rng(123)
    perm = rng.permutation(n)
    n_val = max(1, int(0.15 * n))
    val_idx = perm[:n_val]
    tr_idx = perm[n_val:]

    train_mask = torch.zeros(n, dtype=torch.bool, device=cfg.device)
    val_mask = torch.zeros(n, dtype=torch.bool, device=cfg.device)
    train_mask[tr_idx] = True
    val_mask[val_idx] = True

    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    best = float("inf")
    best_state = None
    bad = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad()
        pred = model(data.x, data.edge_index)
        loss = loss_fn(pred[train_mask], data.y[train_mask], cfg)
        loss.backward()
        if cfg.grad_clip: torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

        model.eval()
        with torch.no_grad():
            pred_val = model(data.x, data.edge_index)[val_mask]
            y_val = data.y[val_mask]
            val_rmse = torch.sqrt(F.mse_loss(pred_val, y_val)).item()

        if val_rmse < best - 1e-6:
            best = val_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience: break

    if best_state is not None: model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict(model: nn.Module, data: Data, device: str) -> np.ndarray:
    model.eval()
    data = data.to(device)
    return model(data.x, data.edge_index).detach().cpu().numpy()

# =========================
# Hyperopt objective INDUCTIVO
# =========================
def make_objective_graphsage_inductive(X_train, y_train, lat_train, lon_train, seed, cv_splits, base_cfg, in_dim):
    kf = KFold(n_splits=cv_splits, shuffle=True, random_state=seed)

    def objective(params: Dict[str, Any]):
        normalizer = params["normalizer"]
        k_graph = int(params["k_graph"])
        hidden_dim = int(params["hidden_dim"])
        num_layers = int(params["num_layers"])
        dropout = float(params["dropout"])
        lr = float(params["lr"])
        weight_decay = float(params["weight_decay"])
        residual = bool(params["residual"])

        fold_mses = []

        for tr_idx, va_idx in kf.split(X_train):
            imputer, x_scaler = make_preprocessor(normalizer)
            X_tr = x_scaler.fit_transform(imputer.fit_transform(X_train[tr_idx])).astype(np.float32)
            X_va = x_scaler.transform(imputer.transform(X_train[va_idx])).astype(np.float32)

            y_scaler = StandardScaler()
            y_tr_log = np.log1p(y_train[tr_idx])
            y_va_log = np.log1p(y_train[va_idx])
            y_tr_scaled = y_scaler.fit_transform(y_tr_log.reshape(-1, 1)).flatten().astype(np.float32)
            y_va_scaled = y_scaler.transform(y_va_log.reshape(-1, 1)).flatten().astype(np.float32)

            lat_tr, lon_tr = lat_train[tr_idx], lon_train[tr_idx]
            lat_va, lon_va = lat_train[va_idx], lon_train[va_idx]

            data_tr = build_pyg_data(X_tr, y_tr_scaled, lat_tr, lon_tr, k_graph=k_graph)
            data_va = build_pyg_data(X_va, y_va_scaled, lat_va, lon_va, k_graph=min(k_graph, len(y_va_scaled)-1 if len(y_va_scaled)>1 else 1))

            cfg = TrainCfg(**{**base_cfg.__dict__, "lr": lr, "weight_decay": weight_decay})
            model = GraphSAGERegressorOptimal(in_dim=in_dim, hidden_dim=hidden_dim, num_layers=num_layers, dropout=dropout, residual=residual)

            model = train_fullbatch_with_es(data_tr, model, cfg)

            pred_va_scaled = predict(model, data_va, cfg.device)
            pred_va_log_rec = y_scaler.inverse_transform(pred_va_scaled.reshape(-1, 1)).flatten()
            pred_va_real = np.expm1(pred_va_log_rec)
            pred_va_real = np.clip(pred_va_real, a_min=0.0, a_max=None)

            y_va_real = y_train[va_idx]
            fold_mses.append(mean_squared_error(y_va_real, pred_va_real))

        return {"loss": float(np.mean(fold_mses)), "status": STATUS_OK, "params": params}
    return objective

# =========================
# Main experiment runner
# =========================
def run_graphsage_inductive(data_path, feature_cols, target_col, lat_col, lon_col, out_xlsx, n_iterations, max_evals, cv_splits, test_size, base_cfg):
    base_cfg = base_cfg or TrainCfg()
    df = pd.read_csv(data_path).reset_index(drop=True)
    X = df[feature_cols].to_numpy()
    y = df[target_col].to_numpy().astype(np.float32)
    lat = df[lat_col].to_numpy().astype(np.float64)
    lon = df[lon_col].to_numpy().astype(np.float64)

    in_dim = X.shape[1]

    space = {
        "normalizer": hp.choice("normalizer", ["standard", "minmax", "robust"]),
        "k_graph": hp.choice("k_graph", [3, 5, 8, 10, 12]),
        "hidden_dim": hp.choice("hidden_dim", [32, 64, 128]),
        "num_layers": hp.choice("num_layers", [2, 3]),
        "dropout": hp.uniform("dropout", 0.1, 0.5),
        "lr": hp.loguniform("lr", np.log(1e-4), np.log(5e-3)),
        "weight_decay": hp.loguniform("weight_decay", np.log(1e-4), np.log(1e-1)),
        "residual": hp.choice("residual", [0, 1]),
    }

    metrics_rows, preds_rows = [], []

    for it in range(n_iterations):
        print(f"\n[GraphSAGE INDUCTIVO] Iteración {it+1}/{n_iterations}")
        set_seed(it)

        X_tr, X_te, y_tr, y_te, lat_tr, lat_te, lon_tr, lon_te = train_test_split(
            X, y, lat, lon, test_size=test_size, random_state=it
        )

        objective = make_objective_graphsage_inductive(
            X_train=X_tr, y_train=y_tr, lat_train=lat_tr, lon_train=lon_tr,
            seed=it, cv_splits=cv_splits, base_cfg=base_cfg, in_dim=in_dim
        )

        trials = Trials()
        best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=max_evals, trials=trials)
        best_params = space_eval(space, best)
        print("  Best params:", best_params)

        imputer, x_scaler = make_preprocessor(best_params["normalizer"])
        X_tr_p = x_scaler.fit_transform(imputer.fit_transform(X_tr)).astype(np.float32)
        X_te_p = x_scaler.transform(imputer.transform(X_te)).astype(np.float32)

        y_scaler = StandardScaler()
        y_tr_log = np.log1p(y_tr)
        y_te_log = np.log1p(y_te)
        y_tr_scaled = y_scaler.fit_transform(y_tr_log.reshape(-1, 1)).flatten().astype(np.float32)
        y_te_scaled = y_scaler.transform(y_te_log.reshape(-1, 1)).flatten().astype(np.float32)

        k_graph = int(best_params["k_graph"])
        data_train = build_pyg_data(X_tr_p, y_tr_scaled, lat_tr, lon_tr, k_graph=k_graph)
        data_test = build_pyg_data(X_te_p, y_te_scaled, lat_te, lon_te, k_graph=min(k_graph, len(y_te_scaled)-1 if len(y_te_scaled)>1 else 1))

        cfg = TrainCfg(**{**base_cfg.__dict__, "lr": float(best_params["lr"]), "weight_decay": float(best_params["weight_decay"])})

        model = GraphSAGERegressorOptimal(
            in_dim=in_dim, hidden_dim=int(best_params["hidden_dim"]), num_layers=int(best_params["num_layers"]),
            dropout=float(best_params["dropout"]), residual=bool(best_params["residual"])
        )

        model = train_fullbatch_with_es(data_train, model, cfg)

        pred_tr_scaled = predict(model, data_train, cfg.device)
        pred_te_scaled = predict(model, data_test, cfg.device)

        pred_tr_log_rec = y_scaler.inverse_transform(pred_tr_scaled.reshape(-1, 1)).flatten()
        pred_te_log_rec = y_scaler.inverse_transform(pred_te_scaled.reshape(-1, 1)).flatten()

        pred_tr = np.expm1(pred_tr_log_rec)
        pred_te = np.expm1(pred_te_log_rec)
        pred_tr = np.clip(pred_tr, a_min=0.0, a_max=None)
        pred_te = np.clip(pred_te, a_min=0.0, a_max=None)

        row = {
            "iteration": it + 1, "model": "GraphSAGE_Inductive_Reg",
            "rmse_train": rmse(y_tr, pred_tr), "mae_train": float(mean_absolute_error(y_tr, pred_tr)),
            "r2_train": float(r2_score(y_tr, pred_tr)), "mape_train": safe_mape(y_tr, pred_tr),
            "rmse_test": rmse(y_te, pred_te), "mae_test": float(mean_absolute_error(y_te, pred_te)),
            "r2_test": float(r2_score(y_te, pred_te)), "mape_test": safe_mape(y_te, pred_te),
            "best_params": json.dumps(best_params, ensure_ascii=False),
        }
        metrics_rows.append(row)

        for j in range(len(y_te)):
            preds_rows.append({
                "iteration": it + 1, "model": "GraphSAGE_Inductive_Reg",
                "lat": float(lat_te[j]), "lon": float(lon_te[j]),
                "y_true": float(y_te[j]), "y_pred": float(pred_te[j]), "residual": float(y_te[j] - pred_te[j]),
            })

        metrics_df, preds_df = pd.DataFrame(metrics_rows), pd.DataFrame(preds_rows)
        with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
            metrics_df.to_excel(writer, sheet_name="metrics", index=False)
            preds_df.to_excel(writer, sheet_name="predictions", index=False)
        print(f"  Train RMSE={row['rmse_train']:.4f} | R2={row['r2_train']:.4f}")
        print(f"  Test RMSE={row['rmse_test']:.4f} | R2={row['r2_test']:.4f}\n")

    print(f"\n Listo. Archivo generado: {out_xlsx}")

if __name__ == "__main__":
    run_graphsage_inductive(
        data_path="/content/large_synthetic_soil_dataset.csv", feature_cols=[f"feat_{i}" for i in range(1, 48)],
        target_col="SOC", lat_col="lat", lon_col="lon", out_xlsx="results_graphsage_inductive_reg.xlsx",
        n_iterations=5, max_evals=20, cv_splits=3, test_size=0.2,
        base_cfg=TrainCfg(epochs=250, patience=30, loss="huber", huber_delta=1.0)
    )

Cargando pipeline GraphSAGE INDUCTIVO (Grafos Aislados + Log-Target + Regularizado)...

[GraphSAGE INDUCTIVO] Iteración 1/5
100%|██████████| 20/20 [01:24<00:00,  4.21s/trial, best loss: 2.6719133059183755]
  Best params: {'dropout': 0.1309479233631882, 'hidden_dim': 32, 'k_graph': 10, 'lr': 0.0014851297479449538, 'normalizer': 'minmax', 'num_layers': 3, 'residual': 1, 'weight_decay': 0.0004150362779399737}
  Train RMSE=1.3491 | R2=0.8334
  Test RMSE=1.4247 | R2=0.8375


[GraphSAGE INDUCTIVO] Iteración 2/5
100%|██████████| 20/20 [01:23<00:00,  4.19s/trial, best loss: 2.293734629948934]
  Best params: {'dropout': 0.2494695184354456, 'hidden_dim': 64, 'k_graph': 12, 'lr': 0.00357450152603494, 'normalizer': 'minmax', 'num_layers': 2, 'residual': 1, 'weight_decay': 0.00026758464049420625}
  Train RMSE=1.3306 | R2=0.8460
  Test RMSE=1.3004 | R2=0.8346


[GraphSAGE INDUCTIVO] Iteración 3/5
100%|██████████| 20/20 [01:14<00:00,  3.73s/trial, best loss: 2.488692839940389]
  Best params: {'dropou

# Enfoque transductivo GraphSage

In [6]:
import os
import json
import random
import multiprocessing
from dataclasses import dataclass
from typing import Dict, Any, List, Optional

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import dropout_edge

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval

torch.set_num_threads(multiprocessing.cpu_count())
print("Cargando pipeline GraphSAGE TRANSDUCTIVO (Jumping Knowledge + Spatial CV + Log-Target)...")

# =========================
# Reproducibility & Metrics
# =========================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def rmse(y_true, y_pred) -> float: return float(np.sqrt(mean_squared_error(y_true, y_pred)))
def safe_mape(y_true, y_pred) -> float:
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mask = y_true != 0
    if not np.any(mask): return float("inf")
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0)

def make_preprocessor(normalizer: str):
    imputer = SimpleImputer(strategy="median")
    if normalizer == "standard": return imputer, StandardScaler()
    elif normalizer == "minmax": return imputer, MinMaxScaler()
    elif normalizer == "robust": return imputer, RobustScaler()
    raise ValueError(normalizer)

# =========================
# Topological Mapping
# =========================
def knn_edge_index_haversine(lat_deg: np.ndarray, lon_deg: np.ndarray, k: int) -> torch.Tensor:
    from sklearn.neighbors import NearestNeighbors
    n = len(lat_deg)
    if n <= 1: return torch.empty((2, 0), dtype=torch.long)
    k_eff = min(k, n - 1)
    coords_rad = np.deg2rad(np.column_stack([lat_deg, lon_deg]))
    nn_model = NearestNeighbors(n_neighbors=k_eff + 1, metric="haversine", algorithm="ball_tree", n_jobs=-1)
    nn_model.fit(coords_rad)
    _, indices = nn_model.kneighbors(coords_rad)
    src, dst = [], []
    for i in range(n):
        for j in indices[i, 1:]:
            src.append(int(j)), dst.append(int(i))
    return torch.tensor([src, dst], dtype=torch.long)

def build_pyg_data_transductive(X_global: np.ndarray, y_global: np.ndarray, lat: np.ndarray, lon: np.ndarray,
                                tr_idx: np.ndarray, va_idx: np.ndarray, k_graph: int) -> Data:
    edge_index = knn_edge_index_haversine(lat, lon, k_graph)
    n = len(y_global)
    train_mask = torch.zeros(n, dtype=torch.bool)
    val_mask = torch.zeros(n, dtype=torch.bool)
    train_mask[tr_idx] = True
    val_mask[va_idx] = True

    return Data(x=torch.from_numpy(X_global.astype(np.float32)), y=torch.from_numpy(y_global.astype(np.float32)), edge_index=edge_index, train_mask=train_mask, val_mask=val_mask)

# =========================
# Model: GraphSAGE Regressor (Con Jumping Knowledge)
# =========================
class GraphSAGERegressorOptimal(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, num_layers: int, dropout: float, residual: bool):
        super().__init__()
        num_layers = max(2, num_layers)
        self.dropout = dropout
        self.residual = residual
        self.edge_dropout_prob = 0.15

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.res_proj = nn.ModuleList() if residual else None

        self.convs.append(SAGEConv(in_dim, hidden_dim))
        self.norms.append(nn.LayerNorm(hidden_dim))
        if residual: self.res_proj.append(nn.Linear(in_dim, hidden_dim))

        for _ in range(1, num_layers):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.norms.append(nn.LayerNorm(hidden_dim))
            if residual: self.res_proj.append(nn.Linear(hidden_dim, hidden_dim))

        # MLP Final con JUMPING KNOWLEDGE
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim + in_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1)
        )

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        if self.training:
            edge_index, _ = dropout_edge(edge_index, p=self.edge_dropout_prob, force_undirected=True)

        h = x
        for li, conv in enumerate(self.convs):
            h_in = h
            h = F.dropout(h, p=self.dropout, training=self.training)
            h = conv(h, edge_index)
            h = self.norms[li](h)
            if self.residual: h = h + self.res_proj[li](h_in)
            h = F.gelu(h)

        h_final = torch.cat([h, x], dim=-1)
        return self.mlp(h_final).squeeze(-1)

# =========================
# Training with Transductive Early Stopping
# =========================
@dataclass
class TrainCfg:
    epochs: int = 300
    patience: int = 35
    lr: float = 1e-3
    weight_decay: float = 1e-4
    loss: str = "huber"
    huber_delta: float = 1.0
    grad_clip: float = 5.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

def loss_fn(pred: torch.Tensor, y: torch.Tensor, cfg: TrainCfg) -> torch.Tensor:
    if cfg.loss == "mse": return F.mse_loss(pred, y)
    if cfg.loss == "huber": return F.smooth_l1_loss(pred, y, beta=cfg.huber_delta)

def train_transductive_with_es(data: Data, model: nn.Module, cfg: TrainCfg) -> nn.Module:
    model = model.to(cfg.device)
    data = data.to(cfg.device)

    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    best = float("inf")
    best_state = None
    bad = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad()
        pred = model(data.x, data.edge_index)
        loss = loss_fn(pred[data.train_mask], data.y[data.train_mask], cfg)
        loss.backward()
        if cfg.grad_clip: torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

        model.eval()
        with torch.no_grad():
            pred_val = model(data.x, data.edge_index)[data.val_mask]
            y_val = data.y[data.val_mask]
            val_rmse = torch.sqrt(F.mse_loss(pred_val, y_val)).item()

        if val_rmse < best - 1e-6:
            best = val_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= cfg.patience: break

    if best_state is not None: model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict_transductive(model: nn.Module, data: Data, idx: np.ndarray, device: str) -> np.ndarray:
    model.eval()
    data = data.to(device)
    idx_tensor = torch.tensor(idx, dtype=torch.long, device=device)
    return model(data.x, data.edge_index)[idx_tensor].detach().cpu().numpy()

# =========================
# Spatial Block Cross-Validation
# =========================
def get_spatial_cv_splits(lat_full, lon_full, tr_idx_master, cv_splits, seed):
    coords = np.column_stack((lat_full[tr_idx_master], lon_full[tr_idx_master]))
    kmeans = KMeans(n_clusters=cv_splits, random_state=seed, n_init='auto')
    cluster_labels = kmeans.fit_predict(coords)
    splits = []
    for cluster_id in range(cv_splits):
        va_rel_idx = np.where(cluster_labels == cluster_id)[0]
        tr_rel_idx = np.where(cluster_labels != cluster_id)[0]
        splits.append((tr_rel_idx, va_rel_idx))
    return splits

# =========================
# Hyperopt objective TRANSDUCTIVO
# =========================
def make_objective_graphsage_transductive(X_full, y_full, lat_full, lon_full, tr_idx_master, seed, cv_splits, base_cfg, in_dim):
    spatial_splits = get_spatial_cv_splits(lat_full, lon_full, tr_idx_master, cv_splits, seed)

    def objective(params: Dict[str, Any]):
        normalizer = params["normalizer"]
        k_graph = int(params["k_graph"])
        hidden_dim = int(params["hidden_dim"])
        num_layers = int(params["num_layers"])
        dropout = float(params["dropout"])
        lr = float(params["lr"])
        weight_decay = float(params["weight_decay"])
        residual = bool(params["residual"])

        fold_mses = []

        for fold_tr_rel, fold_va_rel in spatial_splits:
            real_tr_idx = tr_idx_master[fold_tr_rel]
            real_va_idx = tr_idx_master[fold_va_rel]

            imputer, x_scaler = make_preprocessor(normalizer)
            imputer.fit(X_full[real_tr_idx])
            X_full_imp = imputer.transform(X_full)
            x_scaler.fit(X_full_imp[real_tr_idx])
            X_global_scaled = x_scaler.transform(X_full_imp).astype(np.float32)

            y_scaler = StandardScaler()
            y_full_log = np.log1p(y_full)
            y_scaler.fit(y_full_log[real_tr_idx].reshape(-1, 1))
            y_global_scaled = y_scaler.transform(y_full_log.reshape(-1, 1)).flatten().astype(np.float32)

            data_global = build_pyg_data_transductive(
                X_global=X_global_scaled, y_global=y_global_scaled, lat=lat_full, lon=lon_full,
                tr_idx=real_tr_idx, va_idx=real_va_idx, k_graph=k_graph
            )

            cfg = TrainCfg(**{**base_cfg.__dict__, "lr": lr, "weight_decay": weight_decay})
            model = GraphSAGERegressorOptimal(in_dim=in_dim, hidden_dim=hidden_dim, num_layers=num_layers, dropout=dropout, residual=residual)

            model = train_transductive_with_es(data_global, model, cfg)

            pred_va_scaled = predict_transductive(model, data_global, real_va_idx, cfg.device)
            pred_va_log_rec = y_scaler.inverse_transform(pred_va_scaled.reshape(-1, 1)).flatten()
            pred_va_real = np.expm1(pred_va_log_rec)
            pred_va_real = np.clip(pred_va_real, a_min=0.0, a_max=None)

            y_va_real = y_full[real_va_idx]
            fold_mses.append(mean_squared_error(y_va_real, pred_va_real))

        return {"loss": float(np.mean(fold_mses)), "status": STATUS_OK, "params": params}
    return objective

# =========================
# Main experiment runner
# =========================
def run_graphsage_transductive(data_path, feature_cols, target_col, lat_col, lon_col, out_xlsx, n_iterations, max_evals, cv_splits, test_size, base_cfg):
    base_cfg = base_cfg or TrainCfg()
    df = pd.read_csv(data_path).reset_index(drop=True)
    X = df[feature_cols].to_numpy()
    y = df[target_col].to_numpy().astype(np.float32)
    lat = df[lat_col].to_numpy().astype(np.float64)
    lon = df[lon_col].to_numpy().astype(np.float64)

    in_dim = X.shape[1]
    n_nodes = len(y)
    indices_globales = np.arange(n_nodes)

    space = {
        "normalizer": hp.choice("normalizer", ["standard", "minmax", "robust"]),
        "k_graph": hp.choice("k_graph", [3, 5, 8, 10, 12]),
        "hidden_dim": hp.choice("hidden_dim", [32, 64, 128]),
        "num_layers": hp.choice("num_layers", [2, 3]),
        "dropout": hp.uniform("dropout", 0.1, 0.5),
        "lr": hp.loguniform("lr", np.log(1e-4), np.log(5e-3)),
        "weight_decay": hp.loguniform("weight_decay", np.log(1e-4), np.log(1e-1)),
        "residual": hp.choice("residual", [0, 1]),
    }

    metrics_rows, preds_rows = [], []

    for it in range(n_iterations):
        print(f"\n[GraphSAGE TRANSDUCTIVO REGULARIZADO] Iteración {it+1}/{n_iterations}")
        set_seed(it)

        tr_idx, te_idx = train_test_split(indices_globales, test_size=test_size, random_state=it)

        objective = make_objective_graphsage_transductive(
            X_full=X, y_full=y, lat_full=lat, lon_full=lon, tr_idx_master=tr_idx,
            seed=it, cv_splits=cv_splits, base_cfg=base_cfg, in_dim=in_dim
        )

        trials = Trials()
        best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=max_evals, trials=trials)
        best_params = space_eval(space, best)
        print("  Best params:", best_params)

        imputer, x_scaler = make_preprocessor(best_params["normalizer"])
        imputer.fit(X[tr_idx])
        X_imp = imputer.transform(X)
        x_scaler.fit(X_imp[tr_idx])
        X_global_scaled = x_scaler.transform(X_imp).astype(np.float32)

        y_scaler = StandardScaler()
        y_full_log = np.log1p(y)
        y_scaler.fit(y_full_log[tr_idx].reshape(-1, 1))
        y_global_scaled = y_scaler.transform(y_full_log.reshape(-1, 1)).flatten().astype(np.float32)

        spatial_splits_final = get_spatial_cv_splits(lat, lon, tr_idx, cv_splits=5, seed=it)
        tr_final, va_final = tr_idx[spatial_splits_final[0][0]], tr_idx[spatial_splits_final[0][1]]

        data_global_final = build_pyg_data_transductive(
            X_global=X_global_scaled, y_global=y_global_scaled, lat=lat, lon=lon,
            tr_idx=tr_final, va_idx=va_final, k_graph=int(best_params["k_graph"])
        )

        cfg = TrainCfg(**{**base_cfg.__dict__, "lr": float(best_params["lr"]), "weight_decay": float(best_params["weight_decay"])})

        model = GraphSAGERegressorOptimal(
            in_dim=in_dim, hidden_dim=int(best_params["hidden_dim"]), num_layers=int(best_params["num_layers"]),
            dropout=float(best_params["dropout"]), residual=bool(best_params["residual"])
        )

        model = train_transductive_with_es(data_global_final, model, cfg)

        pred_tr_scaled = predict_transductive(model, data_global_final, tr_idx, cfg.device)
        pred_te_scaled = predict_transductive(model, data_global_final, te_idx, cfg.device)

        pred_tr_log_rec = y_scaler.inverse_transform(pred_tr_scaled.reshape(-1, 1)).flatten()
        pred_te_log_rec = y_scaler.inverse_transform(pred_te_scaled.reshape(-1, 1)).flatten()

        pred_tr = np.expm1(pred_tr_log_rec)
        pred_te = np.expm1(pred_te_log_rec)
        pred_tr = np.clip(pred_tr, a_min=0.0, a_max=None)
        pred_te = np.clip(pred_te, a_min=0.0, a_max=None)

        y_tr_real = y[tr_idx]
        y_te_real = y[te_idx]

        row = {
            "iteration": it + 1, "model": "GraphSAGE_Transductive_Reg",
            "rmse_train": rmse(y_tr_real, pred_tr), "mae_train": float(mean_absolute_error(y_tr_real, pred_tr)),
            "r2_train": float(r2_score(y_tr_real, pred_tr)), "mape_train": safe_mape(y_tr_real, pred_tr),
            "rmse_test": rmse(y_te_real, pred_te), "mae_test": float(mean_absolute_error(y_te_real, pred_te)),
            "r2_test": float(r2_score(y_te_real, pred_te)), "mape_test": safe_mape(y_te_real, pred_te),
            "best_params": json.dumps(best_params, ensure_ascii=False),
        }
        metrics_rows.append(row)

        for j in range(len(te_idx)):
            idx_real = te_idx[j]
            preds_rows.append({
                "iteration": it + 1, "model": "GraphSAGE_Transductive_Reg",
                "lat": float(lat[idx_real]), "lon": float(lon[idx_real]),
                "y_true": float(y_te_real[j]), "y_pred": float(pred_te[j]), "residual": float(y_te_real[j] - pred_te[j]),
            })

        metrics_df, preds_df = pd.DataFrame(metrics_rows), pd.DataFrame(preds_rows)
        with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
            metrics_df.to_excel(writer, sheet_name="metrics", index=False)
            preds_df.to_excel(writer, sheet_name="predictions", index=False)
        print(f"  Train RMSE={row['rmse_train']:.4f} | R2={row['r2_train']:.4f}")
        print(f"  Test RMSE={row['rmse_test']:.4f} | R2={row['r2_test']:.4f}\n")

    print(f"\n Listo. Archivo generado: {out_xlsx}")

if __name__ == "__main__":
    run_graphsage_transductive(
        data_path="/content/large_synthetic_soil_dataset.csv",
        feature_cols=[f"feat_{i}" for i in range(1, 48)],
        target_col="SOC", lat_col="lat", lon_col="lon", out_xlsx="results_graphsage_transductive_reg.xlsx",
        n_iterations=5, max_evals=20, cv_splits=5, test_size=0.2,
        base_cfg=TrainCfg(epochs=250, patience=30, loss="huber", huber_delta=1.0)
    )

Cargando pipeline GraphSAGE TRANSDUCTIVO (Jumping Knowledge + Spatial CV + Log-Target)...

[GraphSAGE TRANSDUCTIVO REGULARIZADO] Iteración 1/5
100%|██████████| 20/20 [02:17<00:00,  6.85s/trial, best loss: 2.117705726623535]
  Best params: {'dropout': 0.37327719303449014, 'hidden_dim': 128, 'k_graph': 5, 'lr': 0.0029921471698252005, 'normalizer': 'standard', 'num_layers': 3, 'residual': 1, 'weight_decay': 0.028768704633133822}
  Train RMSE=1.2314 | R2=0.8612
  Test RMSE=1.4065 | R2=0.8416


[GraphSAGE TRANSDUCTIVO REGULARIZADO] Iteración 2/5
100%|██████████| 20/20 [02:42<00:00,  8.15s/trial, best loss: 2.178764295578003]
  Best params: {'dropout': 0.3012936802876361, 'hidden_dim': 64, 'k_graph': 5, 'lr': 0.0011414516526863942, 'normalizer': 'minmax', 'num_layers': 3, 'residual': 1, 'weight_decay': 0.0031402099930999128}
  Train RMSE=1.3875 | R2=0.8326
  Test RMSE=1.3727 | R2=0.8157


[GraphSAGE TRANSDUCTIVO REGULARIZADO] Iteración 3/5
100%|██████████| 20/20 [02:11<00:00,  6.59s/trial, b